# SMM: Second-Moment Method Acceleration

![Homogeneous reflecting U-235 cube used for the SMM comparison](images/smm_u235_cube.png)

This tutorial compares unaccelerated power iteration with second-moment method (SMM) acceleration for the same multigroup eigenvalue problem. SMM is currently experimental in OpenSn.

## Problem setup

The model is the regression-suite SMM problem: a homogeneous 2 cm cube of U-235 with 84 energy groups and reflecting boundaries. The cross sections are loaded from `test/assets/xs/u235_84g.h5`. Because the domain is homogeneous and closed, it provides a compact correctness check while retaining substantial energy and angular work.

OpenSn's current SMM implementation requires one groupset, P0 scattering, and `save_angular_flux=True`. The saved angular flux supplies the second-moment closure used by the low-order diffusion correction.

In [ ]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature3DXYZ
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.solver import (
    DiscreteOrdinatesProblem,
    PowerIterationKEigenSolver,
    SMMAcceleration,
)
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
repo_root = next(
    path for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs").is_dir()
)

## Build the transport problem

A 5 by 5 by 5 mesh keeps the example compact. The 72-direction product quadrature makes the transport sweeps sufficiently expensive to expose the reduction in work from SMM. Both cases use identical transport settings and two sweeps per power iteration. Run the generated script with four MPI processes.

In [ ]:
def make_problem():
    nodes = [2.0 * i / 5 for i in range(6)]
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes, nodes]
    ).Execute()
    mesh.SetOrthogonalBoundaries()
    mesh.SetUniformBlockID(0)

    xs_u235 = MultiGroupXS()
    xs_u235.LoadFromOpenMC(
        str(repo_root / "test/assets/xs/u235_84g.h5"), "set1", 294.0
    )
    quadrature = GLCProductQuadrature3DXYZ(
        n_polar=6, n_azimuthal=12, scattering_order=0
    )

    return DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=84,
        groupsets=[
            {
                "groups_from_to": (0, 83),
                "angular_quadrature": quadrature,
                "inner_linear_method": "classic_richardson",
                "l_max_its": 2,
                "l_abs_tol": 1.0e-12,
            }
        ],
        xs_map=[{"block_ids": [0], "xs": xs_u235}],
        boundary_conditions=[
            {"name": name, "type": "reflecting"}
            for name in ("xmin", "xmax", "ymin", "ymax", "zmin", "zmax")
        ],
        options={
            "save_angular_flux": True,
            "verbose_inner_iterations": False,
            "verbose_outer_iterations": False,
        },
    )

## Compare power iteration and SMM

The baseline uses no acceleration object. The second case adds `SMMAcceleration` with a continuous piecewise-linear diffusion solve. SMM derives a closure from the angular transport solution, solves the low-order eigenvalue problem, and maps its scalar-flux correction back to transport space.

In [ ]:
def solve(use_smm):
    problem = make_problem()
    solver_options = {
        "problem": problem,
        "k_tol": 1.0e-8,
        "max_iters": 300,
    }
    if use_smm:
        solver_options["acceleration"] = SMMAcceleration(
            problem=problem,
            sdm="pwlc",
            l_abs_tol=1.0e-10,
            max_iters=100,
            pi_max_its=30,
            pi_k_tol=1.0e-8,
        )

    solver = PowerIterationKEigenSolver(**solver_options)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    return (
        solver.GetEigenvalue(),
        solver.GetNumPowerIterations(),
        solver.GetNumSweeps(),
        elapsed,
    )


unaccelerated_k, unaccelerated_iterations, unaccelerated_sweeps, unaccelerated_time = solve(False)
smm_k, smm_iterations, smm_sweeps, smm_time = solve(True)

## Interpret the metrics

Agreement in $k_{\mathrm{eff}}$ is the correctness check. Power iterations and sweeps show how strongly the second-moment correction reduces high-order transport work. Wall time includes construction and solution of the low-order system and is machine-dependent.

In [ ]:
k_difference = abs(smm_k - unaccelerated_k)
speedup = unaccelerated_time / smm_time

if rank == 0:
    print(f"Unaccelerated SMM comparison k-effective={unaccelerated_k:.12e}")
    print(f"SMM k-effective={smm_k:.12e}")
    print(f"SMM k-effective difference={k_difference:.12e}")
    print(f"Unaccelerated SMM comparison power iteration count={unaccelerated_iterations}")
    print(f"SMM power iteration count={smm_iterations}")
    print(f"Unaccelerated SMM comparison sweeps={unaccelerated_sweeps}")
    print(f"SMM sweeps={smm_sweeps}")
    print(f"Unaccelerated SMM comparison wall time (s)={unaccelerated_time:.6f}")
    print(f"SMM wall time (s)={smm_time:.6f}")
    print(f"SMM speedup={speedup:.6f}")

assert k_difference < 1.0e-6
assert smm_iterations < unaccelerated_iterations
assert smm_sweeps < unaccelerated_sweeps

A representative four-process run gives:

| Solve | $k_{\mathrm{eff}}$ | Power iterations | Transport sweeps | Wall time (s) |
|---|---:|---:|---:|---:|
| No acceleration | 2.28043168 | 95 | 190 | 8.014 |
| SMM | 2.28043178 | 2 | 4 | 4.943 |

The eigenvalues differ by $1.0\times10^{-7}$. SMM removes about 98% of the power iterations and transport sweeps and gives a representative speedup of 1.62. The low-order solve has a fixed cost, so timing benefits depend on the size and angular cost of the transport problem.

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()